# 🚨 GCN Suspicious Activity Training Pipeline (Kaggle)

**Prerequisites:** Before running this notebook, upload the following files from your local project into your Kaggle workspace (`/kaggle/working/`) directory by clicking `File -> Upload Data -> New Dataset` or simply dragging the files into the Kaggle IDE file explorer:

1. `tracking_and_pose.py`
2. `data_processing.py`
3. `gcn_model.py`

This notebook walks you through parsing your raw `.mp4` dataset into Tensors, training the network, and exporting the trained `.pth` weight file.

In [ ]:
# 1. Install Required Libraries
!pip install -q ultralytics deep-sort-realtime

In [ ]:
# 2. Imports and Environment Setup
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# Import your custom project files uploaded to Kaggle
from tracking_and_pose import PoseTracker
from data_processing import SkeletonBuffer, GCNPreprocessor
from gcn_model import ActionRecognitionGCN

print("Libraries and local modules imported successfully.")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 3. Feature Extraction: Video to Tensor

# =========================================================================
# IMPORTANT: UPDATE `DATASET_ROOT` TO YOUR KAGGLE DATASET PATH
# Expected Kaggle dataset structure:
# /kaggle/input/dataset/normal/vid1.mp4
# /kaggle/input/dataset/suspicious/vid2.mp4
# =========================================================================
DATASET_ROOT = "/kaggle/input/your-dataset-name" 

def extract_features_from_video(video_path, sequence_length=30):
    tracker = PoseTracker("yolov8n-pose.pt")
    buffer = SkeletonBuffer(max_frames=sequence_length)
    preprocessor = GCNPreprocessor(sequence_length=sequence_length)
    
    seqs = tracker.process_video(video_path=video_path, display=False)
    
    if not seqs:
        return None
        
    longest_id = max(seqs, key=lambda k: len(seqs[k]))
    sequence_data = seqs[longest_id]
    
    for data in sequence_data:
        buffer.update(longest_id, data['keypoints'], data['frame_id'])
        
    raw_tensor = buffer.get_sequence(longest_id)
    if raw_tensor is None:
        return None
        
    formatted_data = preprocessor.process(raw_tensor)
    return formatted_data['joint_data'] 

def construct_dataset(data_dir):
    data_tensors = []
    labels = []
    classes = {"normal": 0, "suspicious": 1}
    
    for class_name, class_id in classes.items():
        search_path = os.path.join(data_dir, class_name, "*.mp4")
        videos = glob.glob(search_path)
        print(f"Found {len(videos)} videos for class '{class_name}'.")
        
        for vid in videos:
            tensor_obj = extract_features_from_video(vid)
            if tensor_obj is not None:
                data_tensors.append(tensor_obj[0]) 
                labels.append(class_id)
                
    if len(data_tensors) == 0:
        print("No videos processed successfully.")
        return None, None
        
    X = np.stack(data_tensors) # Shape: (Total_Videos, 3, 30, 17, 1)
    Y = np.array(labels)
    return X, Y

## UNCOMMENT TO EXTRACT AND COMPILE YOUR REAL DATASET FROM SCRATCH:
# print("Processing videos... this will take some time.")
# X_numpy, Y_numpy = construct_dataset(DATASET_ROOT)
# np.save("X_data.npy", X_numpy)
# np.save("Y_labels.npy", Y_numpy)

In [ ]:
# 4. Neural Network Training Setup

print("Loading Data for Training...")
# Once extracted, you can load dataset rapidly from the saved .npy caches:
# X_numpy = np.load("X_data.npy")
# Y_numpy = np.load("Y_labels.npy")

# --- DUMMY DATA --- 
# Remove these 2 lines once you plug in your real dataset path above
X_numpy = np.random.rand(80, 3, 30, 17, 1).astype(np.float32) 
Y_numpy = np.random.randint(0, 2, 80).astype(np.int64)
# ------------------

X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
Y_tensor = torch.tensor(Y_numpy, dtype=torch.long)

dataset = TensorDataset(X_tensor, Y_tensor)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

model = ActionRecognitionGCN(num_classes=2, in_channels=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

In [ ]:
# 5. The PyTorch Backpropagation Loop
EPOCHS = 15

print("Starting GCN Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100 * correct / total
    
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total if val_total > 0 else 0
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("\n✅ Training Complete!")

# 6. SAVE THE MODEL WEIGHTS
torch.save(model.state_dict(), "pretrained_gcn_weights.pth")
print("\n💾 Saved 'pretrained_gcn_weights.pth' into Kaggle working directory.")
print("Download this file from the Kaggle Output pane on right and drop it into your local project folder to enable live inference!")